In [6]:
import cv2
import cv2.aruco as aruco
import numpy as np


# 1. Define the dictionary and detector parameters
# DICT_6X6_250 contains 250 markers with a 6x6 internal grid
aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_6X6_250)
parameters = aruco.DetectorParameters()

# Initialize the webcam
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
extracted_colours = []

while True:
    ret, frame = cap.read() # Frame contains the image from webcam
    if not ret:
        break
    warped = frame.copy()
    frame_rgb = cv2.cvtColor(frame.copy(), cv2.COLOR_BGR2RGB) # Convert to use camera instead of image on PC
    # 2. Detect the markers
    corners, ids, rejected = aruco.detectMarkers(frame, aruco_dict, parameters=parameters)

    # 3. Draw the results
    if ids is not None:
        src_pts = corners[0].reshape(4,2).astype(np.float32)
        dst_pts =np.float32([[0, 0], [200, 0], [200, 200], [0, 200]])
        matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
        warped = cv2.warpPerspective(frame, matrix, (800, 600))

        x1 = src_pts[0][0]
        x2 = src_pts[1][0]
        y1 = src_pts[0][1]
        y2 = src_pts[1][1]

        cm_per_pixel = 4.5/np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        corrected_distance = cm_per_pixel*np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        x_distance_cm = 4.5+1.4+0.7/2
        x_distance = x_distance_cm/cm_per_pixel
        x_peestick = x1 + x_distance
        x_peestick_cm = (x_peestick-x1)*cm_per_pixel

        y_scaling = 1.5
        vertical_distance_cm =  0.4+0.7 #Spacing + 2 halves of squares
        vertical_distance_pixel = round(vertical_distance_cm/cm_per_pixel)

        leukocytes_y = round(y_scaling*(10+0*vertical_distance_pixel))
        nitrite_y = round(y_scaling*(10+1*vertical_distance_pixel))
        urobilinogen_y = round(y_scaling*(10+2*vertical_distance_pixel))
        protein_y = round(y_scaling*(10+3*vertical_distance_pixel))
        pH_y = round(y_scaling*(10+4*vertical_distance_pixel))
        blood_y = round(y_scaling*(10+5*vertical_distance_pixel))
        sp_gravity_y = round(y_scaling*(10+6*vertical_distance_pixel))
        ketone_y = round(y_scaling*(10+7*vertical_distance_pixel))
        bilirubin_y = round(y_scaling*(10+8*vertical_distance_pixel))
        glucose_y = round(y_scaling*(10+9*vertical_distance_pixel))


        # Check that we hit the correct markers on the pee-stick
        frame_rgb = frame[leukocytes_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[nitrite_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[urobilinogen_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[protein_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[pH_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[blood_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[sp_gravity_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[ketone_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[bilirubin_y, x_peestick]
        extracted_colours.append(frame_rgb)

        frame_rgb = frame[glucose_y, x_peestick]
        extracted_colours.append(frame_rgb)


        aruco.drawDetectedMarkers(frame, corners, ids)

        print("--- Extracted Pad Colors (RGB) ---")
        pad_names =["Leukocytes", "Nitrite", "Urobilinogen", "Protein", "pH", 
                 "Blood", "Sp. Gravity", "Ketone", "Bilirubin", "Glucose"]
                 
        for i, color in enumerate(extracted_colours):
            name = pad_names[i] if i < len(pad_names) else f"Unknown Pad {i+1}"
            print(f"{name:.<15}: {color}")

    cv2.imshow('Pee-Stick Detection', warped)



    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
reference_chart = {
    "Leukocytes": {
        "Negative": [254, 248, 188],
        "15 - Trace": [229, 218, 174],
        "75 - Small": [206, 157, 149],
        "125 - Moderate": [166, 120, 153],
        "500 - Large": [110, 83, 124]
    },
    "Nitrite": {
        "Negative": [253, 250, 222],
        "Positive (+)": [251, 220, 218],
        "Positive (++)": [247, 181, 200],
        "Positive (+++)": [238, 78, 130]
    },
    "Urobilinogen": {
        "3.2 - Normal": [254, 211, 174],
        "16 - Normal": [248, 168, 133],
        "32 - +": [242, 130, 145],
        "64 - ++": [230, 111, 128],
        "128 - +++": [238, 78, 130]
    },
    "Protein": {
        "Negative": [222, 229, 125],
        "Trace - +-": [187, 215, 106],
        "0.3 +": [172, 212, 130],
        "1 = ++": [119, 189, 151],
        "3 = +++": [94, 178, 169],
        ">= 20 = ++++": [0, 148, 149]
    },
    "pH": {
        "5.0": [245, 139, 79],
        "6.0": [249, 165, 85],
        "6.5": [253, 195, 109],
        "7.0": [208, 189, 98],
        "7.5": [136, 148, 85],
        "8.0": [86, 173, 145],
        "8.5": [0, 127, 129]
    },
    "Blood": {
        "Negative": [250, 174, 76],
        "10 - Trace": [207, 161, 65],
        "25 - Small": [161, 156, 84],
        "80 - Moderate": [116, 156, 122],
        "200 - Large": [69, 128, 108]
    },
    "Sp. Gravity": {
        "1.000": [2, 113, 126],
        "1.005": [76, 117, 102],
        "1.010": [123, 136, 105],
        "1.015": [155, 141, 58],
        "1.020": [175, 161, 52],
        "1.025": [197, 167, 48],
        "1.030": [210, 171, 43]
    },
    "Ketone": {
        "Negative": [251, 187, 149],
        "0.5 - Trace": [246, 158, 137],
        "1.5 - Small": [243, 131, 140],
        "4.0 - Moderate": [201, 88, 116],
        "8.0 - Large": [150, 58, 102],
        "16.0 - Large": [120, 41, 90]
    },
    "Bilirubin": {
        "Negative (Baseline)": [253, 250, 222],
        "Negative (Alt)": [253, 223, 144],
        "17 - Small": [251, 187, 131],
        "50 - Moderate": [208, 146, 136],
        "100 - Large": [171, 127, 131]
    },
    "Glucose": {
        "Negative (Blue)": [111, 203, 220],
        "Negative (Green)": [141, 208, 187],
        "5 - Trace": [152, 207, 148],
        "15 - +": [139, 171, 106],
        "30 - ++": [164, 129, 67],
        "60 - +++": [157, 105, 37],
        "110 - ++++": [136, 89, 41]
    }
}

import cv2
import cv2.aruco as aruco
import numpy as np
import matplotlib.pyplot as plt
from skimage import color # Needs: pip install scikit-image

def get_closest_match(extracted_rgb, pad_name, chart):
    if pad_name not in chart:
        return "N/A"
    
    # Convert extracted RGB to LAB
    # We put it in a tiny 1x1 image array because skimage expects images
    rgb_pixels = np.array([[extracted_rgb]], dtype=np.uint8)
    lab_pixel = color.rgb2lab(rgb_pixels)[0][0]
    
    best_label = ""
    min_diff = float('inf')
    
    for label, ref_rgb in chart[pad_name].items():
        # Convert reference RGB to LAB
        ref_rgb_pixels = np.array([[ref_rgb]], dtype=np.uint8)
        ref_lab = color.rgb2lab(ref_rgb_pixels)[0][0]
        
        # Calculate Delta E (CIEDE2000 standard)
        diff = color.deltaE_ciede2000(lab_pixel, ref_lab)
        
        if diff < min_diff:
            min_diff = diff
            best_label = label
            
    return best_label

# --- 2. IMAGE PROCESSING ---
pad_names =["Leukocytes", "Nitrite", "Urobilinogen", "Protein", "pH", 
                 "Blood", "Sp. Gravity", "Ketone", "Bilirubin", "Glucose"]


for i, color in enumerate(extracted_colours):
    name = pad_names[i] if i < len(pad_names) else f"Unknown Pad {i+1}"

    result = get_closest_match(color, name, reference_chart)

    print(f"{name:.<15}: {result}")
        